In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.linear_model import LinearRegression
from scipy import stats
from itertools import combinations

df = pd.read_csv('space_risk_model.csv')
print(f"Shape: {df.shape}")
print(f"Unique satellites: {df['satellite_id'].nunique()}")
print(f"Failure rate: {df['failure_in_12h'].mean():.4f}")

In [ ]:
# Q1: Satellites with max avg solar panel temp per orbit type
sat_avg_temp = df.groupby('satellite_id')['solar_panel_temp_mean'].mean().reset_index()
sat_avg_temp.columns = ['satellite_id', 'avg_solar_temp']
sat_orbit = df[['satellite_id', 'orbit_type']].drop_duplicates()
sat_info = sat_avg_temp.merge(sat_orbit, on='satellite_id')

result_q1 = []
for orbit_type in [0, 1, 2]:
    subset = sat_info[sat_info['orbit_type'] == orbit_type]
    max_sat = subset.loc[subset['avg_solar_temp'].idxmax(), 'satellite_id']
    result_q1.append(int(max_sat))
    print(f"Orbit type {orbit_type}: satellite {max_sat} with avg temp {subset['avg_solar_temp'].max():.4f}")

answer_q1 = f"{result_q1[0]},{result_q1[1]},{result_q1[2]}"
print(f"Answer Q1: {answer_q1}")

In [ ]:
# Q2: Top 5 models by ROC-AUC
model_cols = [f'model_{i}_pred' for i in range(1, 11)]
auc_scores = {}

for col in model_cols:
    auc = roc_auc_score(df['failure_in_12h'], df[col])
    model_num = int(col.split('_')[1])
    auc_scores[model_num] = auc
    print(f"Model {model_num}: AUC = {auc:.6f}")

sorted_models = sorted(auc_scores.items(), key=lambda x: (-x[1], x[0]))
top5_models = sorted_models[:5]

answer_q2_1 = ','.join([str(m[0]) for m in top5_models])
answer_q2_2 = ','.join([str(round(m[1], 2)) for m in top5_models])
print(f"Answer Q2 (models): {answer_q2_1}")
print(f"Answer Q2 (AUC): {answer_q2_2}")

In [ ]:
# Q3: Heating rate ratio
df_sorted = df.sort_values(['satellite_id', 'orbit_number']).reset_index(drop=True)
df_sorted['heating_rate'] = df_sorted.groupby('satellite_id')['solar_panel_temp_mean'].diff(3)

df_valid = df_sorted[df_sorted['heating_rate'].notna()].copy()
no_failure_df = df_valid[df_valid['failure_in_12h'] == 0]
avg_heating_no_failure = no_failure_df['heating_rate'].abs().mean()

failure_orbits = df_sorted[df_sorted['failure_in_12h'] == 1][['satellite_id', 'orbit_number']].copy()

heating_rates_before_failure = []
for idx, row in failure_orbits.iterrows():
    sat_id = row['satellite_id']
    orbit_num = row['orbit_number']
    target_orbit = orbit_num - 3
    
    mask = (df_sorted['satellite_id'] == sat_id) & (df_sorted['orbit_number'] == target_orbit)
    if mask.any():
        hr = df_sorted.loc[mask, 'heating_rate'].values[0]
        if not np.isnan(hr):
            heating_rates_before_failure.append(abs(hr))

avg_heating_before_failure = np.mean(heating_rates_before_failure)
ratio = avg_heating_before_failure / avg_heating_no_failure
answer_q3 = round(ratio)
print(f"Avg |heating_rate| no failure: {avg_heating_no_failure:.6f}")
print(f"Avg |heating_rate| 3 orbits before failure: {avg_heating_before_failure:.6f}")
print(f"Ratio: {ratio:.4f}, Answer Q3: {answer_q3}")

In [ ]:
# Q4: Min FPR at Recall=1.0
fpr_at_recall_1 = {}

for col in model_cols:
    model_num = int(col.split('_')[1])
    predictions = df[col].values
    y_true = df['failure_in_12h'].values
    
    failure_predictions = predictions[y_true == 1]
    t = failure_predictions.min()
    
    y_pred = (predictions >= t).astype(int)
    
    tn = ((y_pred == 0) & (y_true == 0)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    fpr_at_recall_1[model_num] = fpr
    print(f"Model {model_num}: threshold={t:.6f}, FPR={fpr:.4f}")

min_fpr_model = min(fpr_at_recall_1.items(), key=lambda x: x[1])
answer_q4_1 = min_fpr_model[0]
answer_q4_2 = round(min_fpr_model[1], 2)
print(f"Answer Q4: Model {answer_q4_1}, FPR={answer_q4_2}")

In [ ]:
# Q5: Cross-correlation with lag
def calculate_cross_correlation(df, lag):
    correlations = []
    for sat_id in df['satellite_id'].unique():
        sat_data = df[df['satellite_id'] == sat_id].sort_values('orbit_number').copy()
        if lag > 0:
            solar = sat_data['solar_panel_temp_mean'].values[:-lag]
            battery = sat_data['battery_temp_mean'].values[lag:]
        elif lag < 0:
            solar = sat_data['solar_panel_temp_mean'].values[-lag:]
            battery = sat_data['battery_temp_mean'].values[:lag]
        else:
            solar = sat_data['solar_panel_temp_mean'].values
            battery = sat_data['battery_temp_mean'].values
        if len(solar) > 1 and len(battery) > 1:
            if np.std(solar) > 0 and np.std(battery) > 0:
                corr = np.corrcoef(solar, battery)[0, 1]
                correlations.append(corr)
    return np.mean(correlations) if correlations else 0

lags = range(-5, 6)
correlations_by_lag = {}
for lag in lags:
    corr = calculate_cross_correlation(df, lag)
    correlations_by_lag[lag] = corr
    print(f"Lag {lag}: correlation = {corr:.6f}")

max_lag = max(correlations_by_lag.items(), key=lambda x: x[1])
answer_q5_1 = max_lag[0]
answer_q5_2 = round(max_lag[1], 2)
print(f"Answer Q5: lag={answer_q5_1}, corr={answer_q5_2}")

In [ ]:
# Q6: Partial correlations
def partial_correlation(x, y, z):
    r_xy = np.corrcoef(x, y)[0, 1]
    r_xz = np.corrcoef(x, z)[0, 1]
    r_yz = np.corrcoef(y, z)[0, 1]
    numerator = r_xy - r_xz * r_yz
    denominator = np.sqrt((1 - r_xz**2) * (1 - r_yz**2))
    if denominator == 0:
        return 0
    return numerator / denominator

X1 = df['solar_panel_temp_mean'].values
Y1 = df['battery_current_mean'].values
Z1 = df['battery_voltage_mean'].values
X2 = df['battery_current_mean'].values
Y2 = df['battery_voltage_mean'].values
Z2 = df['solar_panel_temp_mean'].values

partial_corr_1 = partial_correlation(X1, Y1, Z1)
partial_corr_2 = partial_correlation(X2, Y2, Z2)
print(f"Partial corr 1: {partial_corr_1:.6f}")
print(f"Partial corr 2: {partial_corr_2:.6f}")

max_partial = max(abs(partial_corr_1), abs(partial_corr_2))
answer_q6 = round(max_partial, 2)
print(f"Answer Q6: {answer_q6}")

In [ ]:
# Q7: Linear risk index - max F1
def calculate_f1_for_weights_fast(w1, w2, w3, df):
    risk_index = (w1 * df['battery_temp_mean'] / 40 + 
                  w2 * df['reaction_wheel_current_mean'] / 2.0 + 
                  w3 * df['attitude_error_mean'] / 1.5)
    y_true = df['failure_in_12h'].values
    thresholds = np.percentile(risk_index, range(1, 100))
    best_f1 = 0
    for t in thresholds:
        y_pred = (risk_index.values > t).astype(int)
        if y_pred.sum() == 0 or y_pred.sum() == len(y_pred):
            continue
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
            best_f1 = max(best_f1, f1)
    return best_f1

best_f1_overall = 0
step = 0.05
for w1 in np.arange(0, 1.01, step):
    for w2 in np.arange(0, 1.01 - w1, step):
        w3 = 1 - w1 - w2
        if w3 < 0:
            continue
        f1 = calculate_f1_for_weights_fast(w1, w2, w3, df)
        if f1 > best_f1_overall:
            best_f1_overall = f1

answer_q7 = round(best_f1_overall, 3)
print(f"Best F1: {best_f1_overall:.6f}, Answer Q7: {answer_q7}")

In [ ]:
# Q8: Anomaly fraction correlation
sat_anomaly_stats = []
for sat_id in df['satellite_id'].unique():
    sat_data = df[df['satellite_id'] == sat_id]
    mu = sat_data['reaction_wheel_current_std'].mean()
    sigma = sat_data['reaction_wheel_current_std'].std(ddof=0)
    threshold = mu + 3 * sigma
    anomaly_count = (sat_data['reaction_wheel_current_std'] > threshold).sum()
    total_count = len(sat_data)
    anomaly_fraction = anomaly_count / total_count
    failure_count = sat_data['failure_in_12h'].sum()
    sat_anomaly_stats.append({
        'satellite_id': sat_id,
        'anomaly_fraction': anomaly_fraction,
        'failure_count': failure_count
    })

stats_df = pd.DataFrame(sat_anomaly_stats)
correlation = np.corrcoef(stats_df['anomaly_fraction'], stats_df['failure_count'])[0, 1]
answer_q8 = round(correlation, 3)
print(f"Correlation: {correlation:.6f}, Answer Q8: {answer_q8}")

In [ ]:
# Q9: Regression residuals std ratio
X = df[['attitude_error_mean', 'reaction_wheel_current_mean']].values
y = df['battery_temp_mean'].values
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)
residuals = y - y_pred
df_with_residuals = df.copy()
df_with_residuals['residual'] = residuals
residuals_failure = df_with_residuals[df_with_residuals['failure_in_12h'] == 1]['residual']
residuals_no_failure = df_with_residuals[df_with_residuals['failure_in_12h'] == 0]['residual']
std_failure = residuals_failure.std()
std_no_failure = residuals_no_failure.std()
ratio_std = std_failure / std_no_failure
answer_q9 = round(ratio_std, 2)
print(f"Std failure: {std_failure:.6f}, Std no failure: {std_no_failure:.6f}")
print(f"Ratio: {ratio_std:.6f}, Answer Q9: {answer_q9}")

In [ ]:
# Q10: Optimal ensemble
def calculate_loss_and_constraints(df, ensemble_pred, t):
    y_true = df['failure_in_12h'].values
    y_pred = (ensemble_pred > t).astype(int)
    tn = ((y_pred == 0) & (y_true == 0)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    loss = 20_000_000 * fn + 100_000 * fp
    total_failures = tp + fn
    fnr_total = fn / total_failures if total_failures > 0 else 0
    warning_rate_total = (tp + fp) / len(y_true)
    constraints_ok = True
    for orbit_type in [0, 1, 2]:
        mask = df['orbit_type'] == orbit_type
        y_true_group = y_true[mask]
        y_pred_group = y_pred[mask]
        tn_g = ((y_pred_group == 0) & (y_true_group == 0)).sum()
        fp_g = ((y_pred_group == 1) & (y_true_group == 0)).sum()
        fn_g = ((y_pred_group == 0) & (y_true_group == 1)).sum()
        tp_g = ((y_pred_group == 1) & (y_true_group == 1)).sum()
        total_failures_g = tp_g + fn_g
        fnr_group = fn_g / total_failures_g if total_failures_g > 0 else 0
        warning_rate_group = (tp_g + fp_g) / len(y_true_group)
        if fnr_group > 0.15:
            constraints_ok = False
        if warning_rate_group > 0.30:
            constraints_ok = False
    if fnr_total > 0.10:
        constraints_ok = False
    if warning_rate_total > 0.25:
        constraints_ok = False
    return loss, constraints_ok, fnr_total, warning_rate_total

best_loss = float('inf')
best_config = None
top_model_indices = [m[0] for m in top5_models]
print(f"Testing combinations: {top_model_indices}")

for combo in combinations(top_model_indices, 3):
    a, b, c = combo
    p_a = df[f'model_{a}_pred'].values
    p_b = df[f'model_{b}_pred'].values
    p_c = df[f'model_{c}_pred'].values
    for w1 in np.arange(0.2, 1.0, 0.2):
        for w2 in np.arange(0.2, 1.0 - w1, 0.2):
            w3 = 1 - w1 - w2
            if w3 < 0.2:
                continue
            ensemble_pred = w1 * p_a + w2 * p_b + w3 * p_c
            thresholds = np.percentile(ensemble_pred, range(50, 95, 5))
            for t in thresholds:
                loss, constraints_ok, fnr, wr = calculate_loss_and_constraints(df, ensemble_pred, t)
                if constraints_ok and loss < best_loss:
                    best_loss = loss
                    best_config = (combo, (w1, w2, w3), t)
                    print(f"New best: models={combo}, t={t:.4f}, loss={loss}")

if best_config:
    models, weights, threshold = best_config
    answer_q10_1 = f"{models[0]},{models[1]},{models[2]}"
    answer_q10_2 = round(threshold, 2)
    answer_q10_3 = round(best_loss)
    print(f"Answer Q10: models={answer_q10_1}, threshold={answer_q10_2}, loss={answer_q10_3}")

In [ ]:
# Save submission
results = [
    {'question_id': 1, 'answer_1': answer_q1, 'answer_2': '', 'answer_3': ''},
    {'question_id': 2, 'answer_1': answer_q2_1, 'answer_2': answer_q2_2, 'answer_3': ''},
    {'question_id': 3, 'answer_1': str(answer_q3), 'answer_2': '', 'answer_3': ''},
    {'question_id': 4, 'answer_1': str(answer_q4_1), 'answer_2': str(answer_q4_2), 'answer_3': ''},
    {'question_id': 5, 'answer_1': str(answer_q5_1), 'answer_2': str(answer_q5_2), 'answer_3': ''},
    {'question_id': 6, 'answer_1': str(answer_q6), 'answer_2': '', 'answer_3': ''},
    {'question_id': 7, 'answer_1': str(answer_q7), 'answer_2': '', 'answer_3': ''},
    {'question_id': 8, 'answer_1': str(answer_q8), 'answer_2': '', 'answer_3': ''},
    {'question_id': 9, 'answer_1': str(answer_q9), 'answer_2': '', 'answer_3': ''},
    {'question_id': 10, 'answer_1': answer_q10_1, 'answer_2': str(answer_q10_2), 'answer_3': str(answer_q10_3)},
]
submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)
print("\nSubmission saved to submission.csv")
print(submission_df)